Load Data & Split Train/Test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Set paths
base_dir = '/content/drive/MyDrive/laby'
excel_path = os.path.join(base_dir, 'Data-Laby 5-12.xlsx')
image_dir = os.path.join(base_dir, 'images')

# Load Excel file
df = pd.read_excel(excel_path)

# Check the column names
print("Available columns:", df.columns)


Available columns: Index(['IMAGE ID', 'LC', 'MD', 'DP', 'TT', 'INDEX GENERAL D'ERREUR',
       'INDEX D'INHIBITION', 'AVERSION DE DELAI'],
      dtype='object')


In [ ]:
# Add filename column (1.jpg to 20.jpg)
df['filename'] = [f"{i}.jpg" for i in range(1, 21)]

# Add full image path
df['filepath'] = df['filename'].apply(lambda x: os.path.join(image_dir, x))

# Keep relevant columns only
df = df[['filepath', 'LC', 'MD', 'DP', 'TT']]


In [ ]:
# 15 train, 5 test
train_df, test_df = train_test_split(df, test_size=5, random_state=42)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
train_df.head()


Train size: 15
Test size: 5


,filepath,LC,MD,DP,TT
5,/content/drive/MyDrive/laby/images/6.jpg,2,2,2,50
11,/content/drive/MyDrive/laby/images/12.jpg,9,4,26,35
3,/content/drive/MyDrive/laby/images/4.jpg,4,3,9,41
18,/content/drive/MyDrive/laby/images/19.jpg,2,1,7,58
16,/content/drive/MyDrive/laby/images/17.jpg,3,5,10,65


Extract Image Features (LC and MD)

Load each image

Convert to grayscale and threshold it

Find the drawn path

Estimate LC and MD from contours

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def extract_features(image_path):
    # Load image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Resize (optional, to standardize input size)
    image = cv2.resize(image, (512, 512))

    # Threshold to binary image
    _, thresh = cv2.threshold(image, 150, 255, cv2.THRESH_BINARY_INV)

    # Find contours (assumed to be the child's path)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # LC: Total length of all contours (approximates path length)
    lc = sum(cv2.arcLength(c, False) for c in contours)

    # MD: Deviation estimate — count of off-path pixels (black pixel density)
    md = np.sum(thresh == 255) / (thresh.shape[0] * thresh.shape[1])  # normalized density

    return lc, md


In [ ]:
# Add LC and MD columns to training dataframe
train_df['LC_pred'] = 0.0
train_df['MD_pred'] = 0.0

for i, row in train_df.iterrows():
    lc, md = extract_features(row['filepath'])
    train_df.at[i, 'LC_pred'] = lc
    train_df.at[i, 'MD_pred'] = md

train_df[['filepath', 'LC', 'LC_pred', 'MD', 'MD_pred']].head()


,filepath,LC,LC_pred,MD,MD_pred
5,/content/drive/MyDrive/laby/images/6.jpg,2,1937.801077,2,0.011623
11,/content/drive/MyDrive/laby/images/12.jpg,9,2362.617311,4,0.012344
3,/content/drive/MyDrive/laby/images/4.jpg,4,1010.722870,3,0.003529
18,/content/drive/MyDrive/laby/images/19.jpg,2,2287.808218,1,0.008007
16,/content/drive/MyDrive/laby/images/17.jpg,3,1275.610172,5,0.004776


let's estimate DP and TT

In [ ]:
# Estimate DP and TT
train_df['DP_pred'] = train_df['LC_pred'] / 20  # adjust this factor if needed
train_df['TT_pred'] = train_df['DP_pred'] * 1.1


Compute INDEX D’ERREUR and INDEX D’INHIBITION

In [ ]:
# Compute predicted INDEX D’ERREUR
train_df['INDEX_ERREUR_pred'] = ((train_df['LC_pred'] + train_df['MD_pred'] + (train_df['DP_pred'] / 10)) / train_df['TT_pred']) * 60

# Compute predicted INDEX D’INHIBITION
train_df['INDEX_INHIBITION_pred'] = ((train_df['DP_pred'] / 10) / train_df['TT_pred']) * 60


Preview Predictions

In [ ]:
train_df[['filepath', 'LC_pred', 'MD_pred', 'DP_pred', 'TT_pred', 'INDEX_ERREUR_pred', 'INDEX_INHIBITION_pred']].head()


,filepath,LC_pred,MD_pred,DP_pred,TT_pred,INDEX_ERREUR_pred,INDEX_INHIBITION_pred
5,/content/drive/MyDrive/laby/images/6.jpg,1937.801077,0.011623,96.890054,106.579059,1096.370180,5.454545
11,/content/drive/MyDrive/laby/images/12.jpg,2362.617311,0.012344,118.130866,129.943952,1096.369336,5.454545
3,/content/drive/MyDrive/laby/images/4.jpg,1010.722870,0.003529,50.536143,55.589758,1096.367445,5.454545
18,/content/drive/MyDrive/laby/images/19.jpg,2287.808218,0.008007,114.390411,125.829452,1096.367454,5.454545
16,/content/drive/MyDrive/laby/images/17.jpg,1275.610172,0.004776,63.780509,70.158559,1096.367721,5.454545


Build a CNN to Predict LC and MD

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.preprocessing import MinMaxScaler
import numpy as np


In [ ]:
IMG_SIZE = 128

def preprocess_image(image_path):
    img = load_img(image_path, color_mode="grayscale", target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img) / 255.0  # Normalize to [0, 1]
    return img


In [ ]:
# Convert image paths to image arrays
X_train = np.array([preprocess_image(path) for path in train_df['filepath']])
X_test = np.array([preprocess_image(path) for path in test_df['filepath']])

# Get the labels (LC and MD)
y_train = train_df[['LC', 'MD']].values
y_test = test_df[['LC', 'MD']].values

# Optional: Normalize labels to 0–1 for better learning
scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(y_train)
y_test_scaled = scaler.transform(y_test)


In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='linear')  # 2 outputs: LC and MD
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,686,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,705,410 (14.14 MB)

 Trainable params: 3,705,410 (14.14 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train, y_train_scaled, epochs=50, batch_size=4,
                    validation_data=(X_test, y_test_scaled))


Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 227ms/step - loss: 5.7174 - mae: 1.5366 - val_loss: 0.7945 - val_mae: 0.7898
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - loss: 0.6867 - mae: 0.6936 - val_loss: 0.2810 - val_mae: 0.4385
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - loss: 0.3286 - mae: 0.5065 - val_loss: 0.1555 - val_mae: 0.3204
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 0.2044 - mae: 0.3595 - val_loss: 0.1474 - val_mae: 0.3139
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step - loss: 0.1478 - mae: 0.3303 - val_loss: 0.1012 - val_mae: 0.2596
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 0.0876 - mae: 0.2549 - val_loss: 0.0965 - val_mae: 0.2520
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - loss: 0.1086 - mae: 0.2631 - val_loss: 0.1399 - val_mae: 0.3027
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - loss: 0.1124 - mae: 0.2694 - val_loss: 0.0930 - val_mae: 0.2461
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - loss: 0.1737 - mae: 

In [ ]:
# Predict on test set
y_pred_scaled = model.predict(X_test)
y_pred = scaler.inverse_transform(y_pred_scaled)

# Compare predictions to true values
for i in range(len(y_test)):
    print(f"Image: {test_df.iloc[i]['filepath'].split('/')[-1]}")
    print(f"True LC: {y_test[i][0]:.2f}, Predicted LC: {y_pred[i][0]:.2f}")
    print(f"True MD: {y_test[i][1]:.2f}, Predicted MD: {y_pred[i][1]:.2f}\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
Image: 1.jpg
True LC: 7.00, Predicted LC: 4.24
True MD: 5.00, Predicted MD: 2.60

Image: 18.jpg
True LC: 6.00, Predicted LC: 4.15
True MD: 4.00, Predicted MD: 2.53

Image: 16.jpg
True LC: 0.00, Predicted LC: 3.96
True MD: 1.00, Predicted MD: 2.46

Image: 2.jpg
True LC: 8.00, Predicted LC: 4.13
True MD: 2.00, Predicted MD: 2.52

Image: 9.jpg
True LC: 3.00, Predicted LC: 4.28
True MD: 2.00, Predicted MD: 2.59



Upload a New Image and Predict LC, MD, INDEX D’ERREUR, and INDEX D’INHIBITION

Define Prediction Function

In [ ]:
def predict_scores(image_path):
    # Preprocess image
    img = preprocess_image(image_path)
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    # Predict LC and MD
    pred_scaled = model.predict(img)
    pred = scaler.inverse_transform(pred_scaled)
    lc_pred, md_pred = pred[0]

    # Estimate DP and TT from LC (same logic as before)
    dp_pred = lc_pred / 20
    tt_pred = dp_pred * 1.1

    # Calculate index d'erreur and d'inhibition
    index_erreur = ((lc_pred + md_pred + (dp_pred / 10)) / tt_pred) * 60
    index_inhibition = ((dp_pred / 10) / tt_pred) * 60

    # Print results
    print(f"📌 Results for: {image_path.split('/')[-1]}")
    print(f"Predicted LC: {lc_pred:.2f}")
    print(f"Predicted MD: {md_pred:.2f}")
    print(f"Estimated DP: {dp_pred:.2f}")
    print(f"Estimated TT: {tt_pred:.2f}")
    print(f"INDEX D'ERREUR: {index_erreur:.2f}")
    print(f"INDEX D'INHIBITION: {index_inhibition:.2f}")


Upload and Test a New Image in Colab

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving 9.jpg to 9.jpg


In [ ]:
new_image_path = list(uploaded.keys())[0]
predict_scores(new_image_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
📌 Results for: 9.jpg
Predicted LC: 4.28
Predicted MD: 2.59
Estimated DP: 0.21
Estimated TT: 0.24
INDEX D'ERREUR: 1755.77
INDEX D'INHIBITION: 5.45


Evaluate the CNN Model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict on test images
y_pred_scaled = model.predict(X_test)
y_pred = scaler.inverse_transform(y_pred_scaled)

# Separate values
true_lc, true_md = y_test[:, 0], y_test[:, 1]
pred_lc, pred_md = y_pred[:, 0], y_pred[:, 1]

# ROUND LC predictions (if required)
pred_lc_rounded = np.round(pred_lc)

# Evaluation metrics for LC
print("📏 LC Evaluation:")
print("MAE:", mean_absolute_error(true_lc, pred_lc_rounded))
print("MSE:", mean_squared_error(true_lc, pred_lc_rounded))
print("R²:", r2_score(true_lc, pred_lc_rounded))

# Evaluation metrics for MD
print("\n📏 MD Evaluation:")
print("MAE:", mean_absolute_error(true_md, pred_md))
print("MSE:", mean_squared_error(true_md, pred_md))
print("R²:", r2_score(true_md, pred_md))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step
📏 LC Evaluation:
MAE: 2.799999952316284
MSE: 9.199999809265137
R²: -0.07476639747619629

📏 MD Evaluation:
MAE: 1.286009669303894
MSE: 2.1312317848205566
R²: 0.013318657875061035
